[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_3_5/16_tag_3_5_eigene_metriken_aktivierungen_losses.ipynb)

# Tag 3-5 - 16 Eigene Metriken, Aktivierungen und Losses

Keras bringt viele Bausteine mit. Manchmal möchte man trotzdem eigene Logik definieren:

- eine eigene Metrik, die im Training angezeigt wird,
- eine eigene Aktivierungsfunktion,
- eine eigene Loss-Funktion.

Dieses Notebook zeigt bewusst einfache Beispiele. Das Ziel ist nicht, bessere Bausteine als Keras zu bauen, sondern die Schnittstellen zu verstehen.


## Lesepfad für dieses Notebook

Die eigenen Bausteine werden nacheinander definiert und später in `model.compile(...)` oder im Modell selbst eingesetzt:

- `leaky_square_relu`: eigene Aktivierung. Wird später in `layers.Dense(..., activation=leaky_square_relu)` benutzt.
- `weighted_binary_crossentropy`: eigener Loss. Wird später in `model.compile(loss=...)` benutzt.
- `RecallAtThreshold`: eigene Metrik-Klasse. Wird später in `model.compile(metrics=[...])` benutzt.

Wenn du also weiter unten einen Namen wieder siehst, stammt er aus einer der Definitionen oben.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True


## Daten

Wir erzeugen ein binäres Klassifikationsproblem mit leicht unausgeglichenen Klassen. Dadurch wird eine eigene Metrik interessant: Accuracy allein kann bei unausgeglichenen Daten täuschen.


In [ ]:
X, y = make_classification(
    n_samples=2200,
    n_features=18,
    n_informative=8,
    n_redundant=4,
    weights=[0.72, 0.28],
    class_sep=1.1,
    flip_y=0.03,
    random_state=RANDOM_STATE,
)
X = StandardScaler().fit_transform(X).astype("float32")
y = y.astype("float32")

X_train, X_rest, y_train, y_rest = train_test_split(
    X, y, test_size=0.35, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.5, random_state=RANDOM_STATE, stratify=y_rest
)

pd.Series(y_train).value_counts(normalize=True).rename("anteil")


## Warum `register_keras_serializable`?

Die Zeile

```python
@keras.utils.register_keras_serializable(package="kurs")
```

ist ein sogenannter Decorator. Er verändert hier nicht die Mathematik der Funktion. Er sagt Keras nur:

> Diese selbst geschriebene Funktion oder Klasse hat einen Namen und darf beim Speichern/Laden wiedergefunden werden.

`package="kurs"` ist dabei nur ein Namensraum. Keras speichert den Baustein intern ungefähr unter einem Namen wie `kurs>leaky_square_relu`. Das hilft, eigene Kursfunktionen von anderen Funktionen mit gleichem Namen zu unterscheiden. Der Name `kurs` ist frei gewählt. In einem echten Projekt könnte dort z. B. `customer_churn` oder `mein_projekt` stehen.

Müssen die Funktionen beim Laden in einer neuen Datei vorher definiert sein?

- **Im gleichen laufenden Notebook:** Wenn die Zellen mit `leaky_square_relu`, `weighted_binary_crossentropy` und `RecallAtThreshold` bereits ausgeführt wurden, kennt Keras die registrierten Bausteine und `load_model(...)` funktioniert ohne `custom_objects`.
- **In einer komplett neuen Python-Datei:** Die Datei muss den Code importieren oder ausführen, der diese Bausteine registriert. Registrierung ist kein magischer Upload in die `.keras`-Datei. Die gespeicherte Datei enthält den Namen und die Konfiguration, aber Python muss die Klasse/Funktion beim Laden kennen.
- **Alternative:** Man kann beim Laden `custom_objects={...}` übergeben. Für Unterricht und Projekte ist Registrierung plus sauberer Import aber übersichtlicher.

Wichtig zur Version: In dieser Kursumgebung nutzen wir `from tensorflow import keras`. Dafür ist `keras.utils.register_keras_serializable(...)` die robuste Variante. `keras.saving.register_keras_serializable(...)` gehört zur separaten Keras-3-API und ist mit `tf.keras` nicht in jeder Installation verfügbar.


## Eigene Aktivierungsfunktion

Eine Aktivierung ist eine Funktion, die auf Tensoren angewendet wird. Hier definieren wir `leaky_square_relu`:

- negative Werte werden wie bei Leaky ReLU schwach durchgelassen,
- positive Werte werden zusätzlich leicht quadratisch verstärkt.

Das ist didaktisch, nicht automatisch eine gute Aktivierung für echte Projekte.


In [ ]:
# Registrierung nur für Speichern/Laden. Die Aktivierung selbst steht in der Funktion.
@keras.utils.register_keras_serializable(package="kurs")
def leaky_square_relu(x):
    positive = tf.nn.relu(x)
    negative = 0.05 * tf.minimum(x, 0.0)
    return positive + 0.03 * tf.square(positive) + negative


xs = tf.linspace(-3.0, 3.0, 200)
plt.plot(xs.numpy(), tf.nn.relu(xs).numpy(), label="relu")
plt.plot(xs.numpy(), leaky_square_relu(xs).numpy(), label="leaky_square_relu")
plt.title("Aktivierungsfunktionen")
plt.legend()


## Eigene Loss-Funktion

Die Loss-Funktion muss differenzierbar sein, weil TensorFlow daraus Gradienten berechnet.

Hier gewichten wir positive Beispiele stärker. Das kann sinnvoll sein, wenn die positive Klasse seltener oder fachlich wichtiger ist.


In [ ]:
# Registrierung nur für Speichern/Laden. Optimiert wird die Funktion unten.
@keras.utils.register_keras_serializable(package="kurs")
def weighted_binary_crossentropy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    weights = 1.0 + 1.5 * y_true
    base_loss = keras.backend.binary_crossentropy(y_true, y_pred)
    return base_loss * weights


## Eigene Metrik als Klasse

Ja: `RecallAtThreshold` ist **nur eine Metrik**. Nach ihr wird nicht optimiert.

Optimiert wird ausschließlich der Loss, den wir in `compile(loss=weighted_binary_crossentropy)` übergeben. Der Optimizer versucht also, diesen Loss kleiner zu machen.

Metriken wie Accuracy, Precision, Recall oder unser `RecallAtThreshold` werden nur berechnet und in `history.history` protokolliert. Sie helfen uns, das Modell zu interpretieren, ändern aber keine Gewichte.

Was macht `RecallAtThreshold` konkret?

- Das Modell gibt Wahrscheinlichkeiten zwischen 0 und 1 aus.
- Standard-Recall nutzt meist Threshold `0.5`: ab 0.5 zählt eine Vorhersage als positiv.
- Unsere Metrik nutzt Threshold `0.35`: schon ab 35 Prozent zählt eine Vorhersage als positiv.
- Dadurch findet man meist mehr positive Fälle, riskiert aber auch mehr False Positives.

Die Formel bleibt Recall:

`Recall = True Positives / (True Positives + False Negatives)`

Eine Keras-Metrik braucht Zustand:

- `update_state(...)`: neue Batch-Ergebnisse einarbeiten,
- `result()`: aktuellen Wert zurückgeben,
- `reset_state()`: Zustand vor der nächsten Epoche zurücksetzen.


In [ ]:
# Registrierung nur für Speichern/Laden der eigenen Metrik-Klasse.
@keras.utils.register_keras_serializable(package="kurs")
class RecallAtThreshold(keras.metrics.Metric):
    def __init__(self, threshold=0.35, name="recall_at_threshold", **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold = threshold

        # Diese beiden Variablen speichern den Zustand der Metrik über mehrere Batches.
        # Sie sind keine Modellgewichte und werden nicht durch Backpropagation gelernt.
        self.true_positives = self.add_weight(name="tp", initializer="zeros")
        self.false_negatives = self.add_weight(name="fn", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        # y_true: echte Labels 0/1.
        # y_pred: Modellwahrscheinlichkeiten zwischen 0 und 1.
        y_true = tf.cast(tf.reshape(y_true, (-1,)), tf.bool)
        y_pred_positive = tf.reshape(y_pred, (-1,)) >= self.threshold

        # True Positive: echtes Label positiv und Vorhersage positiv.
        tp = tf.reduce_sum(tf.cast(tf.logical_and(y_true, y_pred_positive), self.dtype))
        # False Negative: echtes Label positiv, aber Vorhersage nicht positiv.
        fn = tf.reduce_sum(tf.cast(tf.logical_and(y_true, tf.logical_not(y_pred_positive)), self.dtype))

        self.true_positives.assign_add(tp)
        self.false_negatives.assign_add(fn)

    def result(self):
        denominator = self.true_positives + self.false_negatives
        return tf.math.divide_no_nan(self.true_positives, denominator)

    def reset_state(self):
        # Keras ruft das zwischen Epochen auf, damit die nächste Epoche frisch zählt.
        self.true_positives.assign(0.0)
        self.false_negatives.assign(0.0)

    def get_config(self):
        # Damit threshold=0.35 beim Speichern/Laden erhalten bleibt.
        config = super().get_config()
        config.update({"threshold": self.threshold})
        return config


## Modell mit eigenen Bausteinen

Die eigenen Bausteine werden genauso in `Dense`, `compile` und `metrics` genutzt wie eingebaute Keras-Objekte.


In [ ]:
# Hier werden die oben definierten eigenen Bausteine tatsächlich verwendet:
# - leaky_square_relu in der ersten Dense-Schicht
# - weighted_binary_crossentropy als Loss in compile
# - RecallAtThreshold als zusätzliche Metrik in compile
model = keras.Sequential(
    [
        keras.Input(shape=(X_train.shape[1],), name="features"),
        layers.Dense(48, activation=leaky_square_relu),
        layers.Dense(24, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ],
    name="custom_components_model",
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss=weighted_binary_crossentropy,
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall_default_threshold"),
        RecallAtThreshold(threshold=0.35),
    ],
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=35,
    batch_size=64,
    verbose=0,
)

pd.DataFrame(history.history).tail()


In [ ]:
history_df = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history_df[["loss", "val_loss"]].plot(ax=axes[0])
history_df[["val_accuracy", "val_precision", "val_recall_default_threshold", "val_recall_at_threshold"]].plot(ax=axes[1])
axes[0].set_title("Loss")
axes[1].set_title("Validation-Metriken")
axes[1].legend(fontsize=8)


## Laden mit eigenen Bausteinen

Weil wir Aktivierung, Loss und Metrik registriert haben, kann Keras das Modell ohne zusätzliche Angaben laden.

Ohne Registrierung wäre das Modell nicht automatisch kaputt, aber der Ladebefehl müsste die eigenen Bausteine manuell kennen, zum Beispiel über `custom_objects={"leaky_square_relu": leaky_square_relu, ...}`. Für Kurs- und Projektcode ist Registrierung meistens übersichtlicher.


In [ ]:
save_path = "custom_components_model.keras"
model.save(save_path)

# Laden in derselben Notebook-Sitzung funktioniert,
# weil die eigenen Bausteine oben bereits definiert und registriert wurden.
loaded_model = keras.models.load_model(save_path)

# return_dict=True ist robuster als metrics_names manuell zu zippen,
# weil verschiedene Keras-Versionen Metriknamen leicht unterschiedlich ausgeben.
original_result = model.evaluate(X_test, y_test, verbose=0, return_dict=True)
loaded_result = loaded_model.evaluate(X_test, y_test, verbose=0, return_dict=True)

pd.DataFrame(
    [original_result, loaded_result],
    index=["original_model", "loaded_model"],
)


## Limitierungen

- Eigene Loss-Funktionen müssen für die trainierbaren Parameter sinnvoll differenzierbar sein. Harte Schwellenwerte sind in Losses meist problematisch.
- Metriken beeinflussen das Training nicht direkt. Optimiert wird der Loss, nicht die Metrik.
- Ein niedriger Threshold kann Recall erhöhen, aber Precision verschlechtern. Deshalb muss der Threshold fachlich bewertet werden.
- Eigene Aktivierungen können Gradienten explodieren oder verschwinden lassen. Man sollte Trainingskurven und Gradientennormen prüfen.
- Für sauberes Speichern und Laden sollten eigene Funktionen, Klassen und Layer serialisierbar registriert werden.
